> **⚠️ Deprecation Notice**
> This repository is no longer actively maintained. For the latest Gemma examples and tutorials, please visit the [Gemma Cookbook](https://github.com/google-gemma/cookbook).

# Private Fine-Tuning of VaultGemma with LoRA and Differential Privacy

This notebook demonstrates how to fine-tune Google's VaultGemma 1B model on medical data using:
- **LoRA (Low-Rank Adaptation)**: Efficient parameter-efficient fine-tuning
- **4-bit Quantization**: Reduced memory footprint using BitsAndBytes
- **Differential Privacy**: Privacy-preserving training with Opacus

The goal is to create a medical Q&A model while maintaining strong privacy guarantees.
%% [markdown]
## 1. Import Libraries and Load Dataset

We start by importing all necessary libraries:
- `transformers`: For model and tokenizer
- `peft`: For LoRA adapters
- `opacus`: For differential privacy
- `datasets`: For loading and processing the medical dataset

The dataset used is **Medical Meadow Medical Flashcards**, which contains medical question-answer pairs.


In [ ]:
# 1. Install necessary libraries
!pip install -q -U peft accelerate bitsandbytes datasets pandas
!pip install git+https://github.com/huggingface/transformers@v4.56.1-Vault-Gemma-preview
!pip install kagglehub ipywidgets opacus -q

'\n!pip install git+https://github.com/huggingface/transformers@v4.56.1-Vault-Gemma-preview\n! pip install kagglehub\n! pip install ipywidgets\n! pip install protobuf -q\n! pip install tiktoken -q\n! pip install blobfile -q\n! pip install sentencepiece -q\n!pip install -q opacus\n'

In [ ]:
import os
import time
NOTEBOOK_INITIALIZATION_STARTED = time.perf_counter()
import math
import hashlib
import json
import random
import re
import stat
import tempfile
from pathlib import Path

import numpy as np
import torch
import pandas as pd
import kagglehub
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    GemmaTokenizer,
    default_data_collator,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model
from opacus import PrivacyEngine
from opacus.accountants.utils import get_noise_multiplier
from opacus.grad_sample import (
    GradSampleHooksFastGradientClipping,
    GradSampleModule,
    GradSampleModuleExpandedWeights,
    GradSampleModuleFastGradientClipping,
)
from opacus.grad_sample.gsm_base import AbstractGradSampleModule
from opacus.optimizers import DPOptimizer, DPOptimizerFastGradientClipping
from opacus.utils.batch_memory_manager import BatchMemoryManager
from opacus.utils.fast_gradient_clipping_utils import DPLossFastGradientClipping
from opacus.validators import ModuleValidator
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from peft import PeftModel

MODEL_ID = "google/vaultgemma-1b"
DATASET_ID = "medalpaca/medical_meadow_medical_flashcards"
NUM_SAMPLES = 8000
TRAIN_SIZE = 7200
EVAL_SIZE = 800
MAX_SEQUENCE_LENGTH = 256
SEED = 42
ALLOWED_GRAD_SAMPLE_MODES = {"hooks", "functorch", "ew", "ghost"}

def parse_grad_sample_mode(raw_value):
    if not isinstance(raw_value, str) or raw_value not in ALLOWED_GRAD_SAMPLE_MODES:
        raise ValueError("AI_SAFETY_GRAD_SAMPLE_MODE must be exactly one of hooks, functorch, ew, or ghost")
    return raw_value

GRAD_SAMPLE_MODE_ENV = os.environ.get("AI_SAFETY_GRAD_SAMPLE_MODE")
GRAD_SAMPLE_MODE = parse_grad_sample_mode(GRAD_SAMPLE_MODE_ENV)

def parse_compatibility_attempt_id(raw_value):
    if not isinstance(raw_value, str) or re.fullmatch(r"[a-z0-9][a-z0-9_-]{0,63}", raw_value) is None:
        raise ValueError("AI_SAFETY_COMPATIBILITY_ATTEMPT_ID must match [a-z0-9][a-z0-9_-]{0,63}")
    return raw_value

COMPATIBILITY_ATTEMPT_ID_ENV = os.environ.get("AI_SAFETY_COMPATIBILITY_ATTEMPT_ID")
COMPATIBILITY_ATTEMPT_ID = parse_compatibility_attempt_id(COMPATIBILITY_ATTEMPT_ID_ENV)

def require_contract(condition, message):
    if not condition:
        raise AssertionError(message)

def save_private_adapter_checkpoint(private_module, tokenizer_to_save, checkpoint_path):
    allowed_wrapper_types = (
        GradSampleModule,
        GradSampleModuleExpandedWeights,
        GradSampleModuleFastGradientClipping,
    )
    if not isinstance(private_module, AbstractGradSampleModule) or type(private_module) not in allowed_wrapper_types:
        raise TypeError("expected a current Opacus private-module wrapper")
    if tuple(private_module._modules) != ("_module",):
        raise TypeError("Opacus wrapper must contain exactly one inner _module")
    inner_module = private_module._module
    if isinstance(inner_module, AbstractGradSampleModule):
        raise TypeError("nested Opacus private-module wrappers are forbidden")
    model_save = getattr(inner_module, "save_pretrained", None)
    tokenizer_save = getattr(tokenizer_to_save, "save_pretrained", None)
    if not callable(model_save):
        raise TypeError("inner private module must provide callable save_pretrained")
    if not callable(tokenizer_save):
        raise TypeError("tokenizer must provide callable save_pretrained")
    model_save(checkpoint_path)
    tokenizer_save(checkpoint_path)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

SELECTED_GPU_INDEX = int(os.environ.get("VAULTGEMMA_GPU_INDEX", "0"))
if not torch.cuda.is_available():
    raise RuntimeError("VaultGemma training requires one CUDA GPU")
if not 0 <= SELECTED_GPU_INDEX < torch.cuda.device_count():
    raise ValueError(f"invalid GPU index: {SELECTED_GPU_INDEX}")
torch.cuda.set_device(SELECTED_GPU_INDEX)
DEVICE = torch.device(f"cuda:{SELECTED_GPU_INDEX}")

def build_dp_noise_generator(device, seed):
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)
    return generator

DP_NOISE_GENERATOR = build_dp_noise_generator(DEVICE, SEED)

# Deterministically shuffle the complete source split before selecting 8,000 records.
medical_data = load_dataset(DATASET_ID, split="train")
if len(medical_data) < NUM_SAMPLES:
    raise ValueError(f"dataset has {len(medical_data)} records; {NUM_SAMPLES} required")
source_fingerprint = medical_data._fingerprint
indexed_medical_data = medical_data.add_column("_source_index", list(range(len(medical_data))))
shuffled_medical_data = indexed_medical_data.shuffle(seed=SEED)
selected_dataset = shuffled_medical_data.select(range(NUM_SAMPLES))
require_contract(len(selected_dataset) == NUM_SAMPLES, "selected dataset size mismatch")

## 2. Load Base Model with 4-bit Quantization

We load **VaultGemma 1B** from Kaggle with 4-bit quantization to reduce memory usage:
- **NF4 quantization**: Normal Float 4-bit quantization for optimal quality
- **Double quantization**: Further compression by quantizing the quantization constants
- **bfloat16 compute**: Uses brain floating point for stable training

The model is automatically distributed across available GPUs using `device_map="auto"`.

In [3]:
# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map={"": SELECTED_GPU_INDEX},
)

# Load tokenizer
tokenizer = GemmaTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

## 3. Apply LoRA Adapters

**LoRA (Low-Rank Adaptation)** adds small trainable matrices to specific layers while keeping the base model frozen:
- **r=8**: Rank of the low-rank matrices (higher = more capacity but more parameters)
- **lora_alpha=16**: Scaling factor for LoRA weights
- **target_modules**: Which attention and MLP layers to adapt (all projection layers in Gemma)
- **lora_dropout=0.05**: Dropout for regularization

This approach trains only ~1-2% of the total parameters, making training much faster and memory-efficient.


In [4]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA adapters to the model (for new training)
peft_model = get_peft_model(model, lora_config)

EXPECTED_TOTAL_PARAMETERS = 1_045_583_488
EXPECTED_TRAINABLE_PARAMETERS = 6_842_368
EXPECTED_TRAINABLE_PERCENT = 0.6544066618
EXPECTED_PACKED_PARAMETER_ELEMENTS = 673_699_456
EXPECTED_PACKED_STORAGE_BYTES = 989_199_616
# PEFT corrects packed Params4bit storage counts to logical model parameters.
trainable_parameters, total_parameters = peft_model.get_nb_trainable_parameters()
packed_parameter_elements = sum(parameter.numel() for parameter in peft_model.parameters())
packed_storage_bytes = sum(
    parameter.numel() * parameter.element_size() for parameter in peft_model.parameters()
)
trainable_percent = 100.0 * trainable_parameters / total_parameters
require_contract(
    total_parameters == EXPECTED_TOTAL_PARAMETERS,
    f"total parameter mismatch: {total_parameters} != {EXPECTED_TOTAL_PARAMETERS}",
)
require_contract(
    trainable_parameters == EXPECTED_TRAINABLE_PARAMETERS,
    f"trainable parameter mismatch: {trainable_parameters} != {EXPECTED_TRAINABLE_PARAMETERS}",
)
require_contract(
    abs(trainable_percent - EXPECTED_TRAINABLE_PERCENT) <= 1e-6,
    f"trainable ratio mismatch: {trainable_percent} != {EXPECTED_TRAINABLE_PERCENT}",
)
require_contract(
    packed_parameter_elements == EXPECTED_PACKED_PARAMETER_ELEMENTS,
    f"packed parameter element mismatch: {packed_parameter_elements} != {EXPECTED_PACKED_PARAMETER_ELEMENTS}",
)
require_contract(
    packed_storage_bytes == EXPECTED_PACKED_STORAGE_BYTES,
    f"packed storage byte mismatch: {packed_storage_bytes} != {EXPECTED_PACKED_STORAGE_BYTES}",
)

print("Model and LoRA adapters loaded for training!")
peft_model.print_trainable_parameters()

# Set model to training mode
peft_model.train()

Model and LoRA adapters loaded for training!
trainable params: 6,842,368 || all params: 1,045,583,488 || trainable%: 0.6544


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): VaultGemmaForCausalLM(
      (model): VaultGemmaModel(
        (embed_tokens): Embedding(256000, 1152, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x VaultGemmaDecoderLayer(
            (self_attn): VaultGemmaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1152, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1152, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1024, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
          

## 4. Prepare and Tokenize Dataset

We create a custom tokenization function that:
1. Formats each example as an instruction-following prompt
2. Tokenizes the full text (question + answer)
3. **Masks the prompt tokens** in the labels by setting them to -100

This ensures the model only learns to generate the **response**, not to repeat the question.
The masking prevents the loss function from penalizing the model for the input prompt.


In [ ]:
def tokenize_and_mask(samples):
    """
    Tokenizes the input and output, then masks the prompt tokens in labels
    so that the model only learns to predict the response.
    """
    # Format prompts and responses
    full_prompts = [
        f"Instruction:\nAnswer this question truthfully.\n\nQuestion:\n{inp}" 
        for inp in samples["input"]
    ]
    responses = [f"\n\nResponse:\n{out}" for out in samples["output"]]
    
    # Tokenize full text (prompt + response)
    model_inputs = tokenizer(
        [p + r for p, r in zip(full_prompts, responses)],
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )
    
    # Tokenize only prompts to determine their length
    prompt_tokens = tokenizer(
        full_prompts,
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )
    
    # Create labels (copy of input_ids)
    labels = model_inputs["input_ids"].clone()
    
    # Mask prompt tokens in labels (set to -100 so they're ignored in loss calculation)
    for i in range(len(labels)):
        prompt_len = int(prompt_tokens["attention_mask"][i].sum())
        labels[i][:prompt_len] = -100

    # Padding is not a response target, even when pad_token equals eos_token.
    labels[model_inputs["attention_mask"].eq(0)] = -100
    
    model_inputs["labels"] = labels
    return model_inputs

def validate_response_only_batch(batch):
    required = {"input_ids", "attention_mask", "labels"}
    missing = required.difference(batch)
    if missing:
        raise ValueError(f"response-only batch is missing fields: {sorted(missing)}")
    labels = batch["labels"]
    attention_mask = batch["attention_mask"]
    if labels.ndim != 2 or labels.shape != attention_mask.shape:
        raise ValueError("labels and attention_mask must have matching [batch, sequence] shapes")
    if torch.any(labels[attention_mask.eq(0)].ne(-100)):
        raise ValueError("padding labels must be -100")
    for row_index in range(labels.shape[0]):
        active_labels = labels[row_index][attention_mask[row_index].bool()]
        response_mask = active_labels.ne(-100)
        if not torch.any(active_labels.eq(-100)):
            raise ValueError(f"record {row_index} has no masked prompt tokens")
        if not torch.any(response_mask):
            raise ValueError("every record must contain at least one response target token")
        first_response = int(torch.nonzero(response_mask, as_tuple=False)[0, 0])
        if torch.any(active_labels[first_response:].eq(-100)):
            raise ValueError(f"record {row_index} has a non-prefix prompt mask")
    return batch

def response_only_data_collator(features):
    prepared_labels = torch.stack([torch.as_tensor(feature["labels"]) for feature in features])
    batch = default_data_collator(features)
    if not torch.equal(batch["labels"], prepared_labels):
        raise ValueError("collator changed prepared response-only labels")
    return validate_response_only_batch(batch)

def shift_response_tensors(logits, labels):
    if logits.ndim != 3 or labels.ndim != 2:
        raise ValueError("logits and labels must have [batch, sequence, vocab] and [batch, sequence] shapes")
    if logits.shape[:2] != labels.shape:
        raise ValueError("logits and labels sequence shapes do not match")
    shifted_logits = logits[:, :-1, :].contiguous()
    shifted_labels = labels[:, 1:].contiguous()
    counts = shifted_labels.ne(-100).sum(dim=1)
    if torch.any(counts == 0):
        raise ValueError("every record must contain at least one response target token")
    return shifted_logits, shifted_labels

def response_only_loss(logits, labels):
    shifted_logits, shifted_labels = shift_response_tensors(logits, labels)
    token_losses = torch.nn.functional.cross_entropy(
        shifted_logits.float().reshape(-1, shifted_logits.size(-1)),
        shifted_labels.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(shifted_labels.shape)
    counts = shifted_labels.ne(-100).sum(dim=1)
    return (token_losses.sum(dim=1) / counts).mean()

def ghost_response_only_loss(ghost_criterion, module, optimizer, logits, labels):
    if ghost_criterion is None:
        raise RuntimeError("ghost mode requires the criterion returned by PrivacyEngine.make_private")
    if not isinstance(ghost_criterion, DPLossFastGradientClipping):
        raise TypeError("ghost criterion must be Opacus DPLossFastGradientClipping")
    fast_module_types = (
        GradSampleModuleFastGradientClipping,
        GradSampleHooksFastGradientClipping,
    )
    if not isinstance(module, fast_module_types):
        raise TypeError("ghost mode requires an Opacus fast-gradient-clipping module")
    for capability in ("get_clipping_coef", "disable_hooks", "enable_hooks"):
        if not callable(getattr(module, capability, None)):
            raise TypeError(f"ghost module lacks required capability: {capability}")
    if not isinstance(optimizer, DPOptimizerFastGradientClipping):
        raise TypeError("ghost mode requires DPOptimizerFastGradientClipping")
    if ghost_criterion.module is not module or ghost_criterion.optimizer is not optimizer:
        raise ValueError("ghost criterion is not bound to the live module and optimizer")
    inner_criterion = ghost_criterion.criterion
    if not isinstance(inner_criterion, torch.nn.CrossEntropyLoss):
        raise TypeError("ghost criterion must wrap CrossEntropyLoss")
    if inner_criterion.ignore_index != -100 or inner_criterion.reduction != "mean":
        raise ValueError("ghost CrossEntropyLoss must use ignore_index=-100 and reduction='mean'")
    shifted_logits, shifted_labels = shift_response_tensors(logits, labels)
    # Validate before Opacus applies clamp(min=1) to ignored-token counts.
    if torch.any(shifted_labels.ne(-100).sum(dim=1) == 0):
        raise ValueError("every record must contain at least one response target token")
    return ghost_criterion(
        shifted_logits.float().reshape(-1, shifted_logits.size(-1)),
        shifted_labels.reshape(-1),
        shape=shifted_logits.shape,
    )

def dispatch_response_only_loss(grad_sample_mode, ghost_criterion, module, optimizer, logits, labels):
    if grad_sample_mode == "ghost":
        return ghost_response_only_loss(ghost_criterion, module, optimizer, logits, labels)
    if grad_sample_mode in {"hooks", "functorch", "ew"}:
        return response_only_loss(logits, labels)
    raise ValueError(f"unsupported gradient-sample mode: {grad_sample_mode}")

def forward_model_inputs(module, batch):
    if "input_ids" not in batch:
        raise ValueError("model batch is missing input_ids")
    model_kwargs = {
        key: value for key, value in batch.items()
        if key not in {"input_ids", "labels"}
    }
    return module(batch["input_ids"], **model_kwargs)

def forward_response_only_loss(
    module, batch, optimizer, grad_sample_mode, training, ghost_loss_criterion=None
):
    if "labels" not in batch:
        raise ValueError("response-only batch is missing labels")
    outputs = forward_model_inputs(module, batch)
    if training:
        return dispatch_response_only_loss(
            grad_sample_mode, ghost_loss_criterion, module, optimizer,
            outputs.logits, batch["labels"]
        )
    return response_only_loss(outputs.logits, batch["labels"])

def accumulate_record_weighted_loss(loss_sum, record_count, batch_mean, batch_size):
    if batch_size <= 0:
        raise ValueError("batch size must be positive")
    return loss_sum + float(batch_mean) * batch_size, record_count + batch_size

def finalize_record_weighted_loss(loss_sum, record_count):
    if record_count <= 0:
        raise ValueError("record count must be positive")
    return loss_sum / record_count

def optimizer_completed_logical_step(optimizer):
    return not bool(getattr(optimizer, "_is_last_step_skipped", False))

def resolve_project_root():
    configured_root = os.environ.get("AI_SAFETY_PROJECT_ROOT")
    candidate = (
        Path(configured_root).expanduser().resolve()
        if configured_root
        else Path.cwd().resolve()
    )
    required_markers = (
        Path("task_plan.md"),
        Path("workspaces/vaultgemma_vectorized/upstream_manifest.json"),
    )
    missing_markers = [str(marker) for marker in required_markers if not (candidate / marker).is_file()]
    if missing_markers:
        source = "AI_SAFETY_PROJECT_ROOT" if configured_root else "current working directory"
        raise RuntimeError(
            f"{source} is not the verified project root {candidate}; "
            f"missing markers: {missing_markers}"
        )
    return candidate

def publish_bytes_no_replace(destination, payload, before_publish=None):
    destination = Path(destination)
    if not isinstance(payload, bytes):
        raise TypeError("payload must be bytes")
    destination.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{destination.name}.", suffix=".tmp", dir=destination.parent
    )
    temporary_path = Path(temporary_name)
    try:
        stream = os.fdopen(descriptor, "wb")
        descriptor = -1
        with stream:
            stream.write(payload)
            stream.flush()
            os.fsync(stream.fileno())
        if before_publish is not None:
            before_publish(temporary_path)
        try:
            os.link(temporary_path, destination)
        except FileExistsError as error:
            if not hasattr(os, "O_NOFOLLOW"):
                raise RuntimeError(
                    "safe existing-manifest inspection requires O_NOFOLLOW"
                )
            existing_flags = os.O_RDONLY | os.O_NONBLOCK | os.O_NOFOLLOW
            existing_descriptor = -1
            try:
                existing_descriptor = os.open(destination, existing_flags)
                destination_stat = os.fstat(existing_descriptor)
                if not stat.S_ISREG(destination_stat.st_mode):
                    raise FileExistsError(
                        f"refusing nonregular existing manifest: {destination}"
                    )
                destination_chunks = []
                while True:
                    chunk = os.read(existing_descriptor, 1024 * 1024)
                    if not chunk:
                        break
                    destination_chunks.append(chunk)
                destination_payload = b"".join(destination_chunks)
            except FileExistsError:
                raise
            except OSError as inspection_error:
                raise FileExistsError(
                    f"refusing to inspect or replace existing manifest: {destination}"
                ) from inspection_error
            finally:
                if existing_descriptor >= 0:
                    os.close(existing_descriptor)
            if destination_payload != payload:
                raise FileExistsError(
                    f"refusing to overwrite mismatched manifest: {destination}"
                ) from error
        else:
            directory_flags = os.O_RDONLY | getattr(os, "O_DIRECTORY", 0)
            directory_descriptor = os.open(destination.parent, directory_flags)
            try:
                os.fsync(directory_descriptor)
            finally:
                os.close(directory_descriptor)
    finally:
        if descriptor >= 0:
            os.close(descriptor)
        try:
            temporary_path.unlink()
        except FileNotFoundError:
            pass

# Persist selection provenance without storing medical text in the manifest.
raw_train_dataset = selected_dataset.select(range(TRAIN_SIZE))
raw_eval_dataset = selected_dataset.select(range(TRAIN_SIZE, NUM_SAMPLES))
require_contract(len(raw_train_dataset) == TRAIN_SIZE, "train split size mismatch")
require_contract(len(raw_eval_dataset) == EVAL_SIZE, "eval split size mismatch")

def record_sha256(record):
    content = {
        key: record[key] for key in sorted(record) if key != "_source_index"
    }
    payload = json.dumps(
        content, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

selected_source_indices = list(selected_dataset["_source_index"])
selected_record_sha256 = [record_sha256(record) for record in selected_dataset]
selected_content_sha256 = hashlib.sha256(
    "\n".join(selected_record_sha256).encode("ascii")
).hexdigest()
print(json.dumps({
    "selected_size": len(selected_dataset),
    "selected_fingerprint": selected_dataset._fingerprint,
    "first_source_index": selected_source_indices[0],
    "selected_content_sha256": selected_content_sha256,
}, sort_keys=True))
dataset_manifest = {
    "dataset_id": DATASET_ID,
    "seed": SEED,
    "source_size": len(medical_data),
    "selected_size": NUM_SAMPLES,
    "train_size": TRAIN_SIZE,
    "eval_size": EVAL_SIZE,
    "source_fingerprint": source_fingerprint,
    "shuffled_fingerprint": shuffled_medical_data._fingerprint,
    "selected_fingerprint": selected_dataset._fingerprint,
    "selected_source_indices": selected_source_indices,
    "train_source_indices": selected_source_indices[:TRAIN_SIZE],
    "eval_source_indices": selected_source_indices[TRAIN_SIZE:],
    "selected_record_sha256": selected_record_sha256,
    "selected_content_sha256": selected_content_sha256,
}
project_root = resolve_project_root()
dataset_manifest_path = project_root / "results/vaultgemma/dataset_manifest.json"
serialized_dataset_manifest = json.dumps(
    dataset_manifest, ensure_ascii=False, indent=2, sort_keys=True
) + "\n"
publish_bytes_no_replace(
    dataset_manifest_path, serialized_dataset_manifest.encode("utf-8")
)

# Apply tokenization function
tokenized_dataset = selected_dataset.map(
    tokenize_and_mask,
    batched=True,
    remove_columns=selected_dataset.column_names
)

## 5. Configure Training Parameters

We set up all training hyperparameters and create data loaders:
- **90/10 train/validation split** for monitoring overfitting
- **Batch size = 1** with **gradient accumulation = 8** (effective batch size of 8)
- **Learning rate = 2e-5** with cosine decay schedule
- **20 epochs** of training

The small batch size is necessary due to memory constraints from the quantized model.


In [ ]:
# Split dataset into train and validation
train_dataset = tokenized_dataset.select(range(TRAIN_SIZE))
eval_dataset = tokenized_dataset.select(range(TRAIN_SIZE, NUM_SAMPLES))
require_contract(len(train_dataset) == TRAIN_SIZE, "tokenized train size mismatch")
require_contract(len(eval_dataset) == EVAL_SIZE, "tokenized eval size mismatch")

# Training hyperparameters
NUM_TRAIN_EPOCHS = 6
LOGICAL_BATCH_SIZE = 128
MAX_PHYSICAL_BATCH_SIZE = 16
EVAL_BATCH_SIZE = MAX_PHYSICAL_BATCH_SIZE
TOTAL_OPTIMIZER_STEPS = 342
GRADIENT_ACCUMULATION_STEPS = 1
OPTIMIZER_NAME = "AdamW"
LEARNING_RATE = 1e-4
SCHEDULER_NAME = "cosine"
NUM_WARMUP_STEPS = 5
GRADIENT_CHECKPOINTING = False
device = DEVICE
num_train_epochs = NUM_TRAIN_EPOCHS
per_device_train_batch_size = LOGICAL_BATCH_SIZE
gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS
learning_rate = LEARNING_RATE
eval_steps = 100
logging_steps = 40
if GRADIENT_CHECKPOINTING:
    peft_model.gradient_checkpointing_enable()
else:
    peft_model.gradient_checkpointing_disable()

# The default collator stacks and preserves the response-only labels prepared above.
data_collator = response_only_data_collator
# Task 5 make_private sets this from its exact four-value ghost return.
GHOST_LOSS_CRITERION = None

# Create data loaders
DATA_LOADER_GENERATOR = torch.Generator()
DATA_LOADER_GENERATOR.manual_seed(SEED)
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=per_device_train_batch_size, 
    shuffle=True,
    collate_fn=data_collator,
    generator=DATA_LOADER_GENERATOR,
    drop_last=False
)
eval_dataloader = DataLoader(
    eval_dataset, 
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    drop_last=False
)

## 6. Apply Differential Privacy with Opacus

**Differential Privacy (DP)** ensures that the trained model doesn't memorize specific training examples:
- **ε (epsilon) = 8.0**: Privacy budget (lower = more privacy, but potentially worse performance)
- **δ (delta) = 1e-5**: Probability of privacy breach (should be < 1/dataset_size)
- **max_grad_norm = 1.0**: Clips gradients to prevent any single example from having too much influence

Opacus modifies the training loop to add calibrated noise to gradients, providing mathematical privacy guarantees.
The final epsilon value will tell us exactly how much privacy was consumed during training.


In [ ]:
# Differential privacy setup. Epsilon has no implicit default: every run selects it.
TARGET_DELTA = 1e-5
MAX_GRAD_NORM = 1.0
ALLOWED_TARGET_EPSILONS = {0.5, 2.0, 8.0}
EXPECTED_NOISE_MULTIPLIERS = {
    0.5: 2.578125,
    2.0: 1.015625,
    8.0: 0.6005859375,
}
EPSILON_CALIBRATION_TOLERANCE = 0.01
Q = LOGICAL_BATCH_SIZE / TRAIN_SIZE
EXPECTED_FULL_LOGICAL_BATCHES_PER_EPOCH = 56
EXPECTED_REMAINDER_RECORDS_PER_EPOCH = 32
EXPECTED_LOGICAL_STEPS_PER_EPOCH = 57
EXPECTED_PHYSICAL_MICROBATCHES_PER_EPOCH = 450
EXPECTED_TOTAL_PHYSICAL_MICROBATCHES = 2700

def parse_target_epsilon(raw_value):
    if raw_value is None:
        raise ValueError("VAULTGEMMA_TARGET_EPSILON must select one of 0.5, 2, or 8")
    try:
        epsilon_value = float(raw_value)
    except (TypeError, ValueError) as error:
        raise ValueError("VAULTGEMMA_TARGET_EPSILON must be numeric") from error
    if epsilon_value not in ALLOWED_TARGET_EPSILONS:
        raise ValueError(
            f"unsupported target epsilon {raw_value!r}; choose 0.5, 2, or 8"
        )
    return epsilon_value

def accountant_logical_steps(accountant):
    return sum(int(history_entry[2]) for history_entry in accountant.history)

def validate_accountant_history(accountant, expected_noise_multiplier, expected_sample_rate, expected_steps):
    history = list(accountant.history)
    if not history:
        raise AssertionError("accountant history must not be empty")
    accumulated_steps = 0
    for entry in history:
        if len(entry) != 3:
            raise AssertionError(f"invalid accountant history entry: {entry!r}")
        noise_multiplier, sample_rate, steps = entry
        if noise_multiplier != expected_noise_multiplier:
            raise AssertionError(f"accountant noise multiplier mismatch: {entry!r}")
        if sample_rate != expected_sample_rate:
            raise AssertionError(f"accountant sample rate mismatch: {entry!r}")
        accumulated_steps += int(steps)
    if accumulated_steps != expected_steps:
        raise AssertionError(f"accountant step count mismatch: {accumulated_steps}")

def fully_qualified_class_name(instance):
    instance_type = type(instance)
    return f"{instance_type.__module__}.{instance_type.__qualname__}"

def collect_trainable_module_types(module):
    module_types = {
        fully_qualified_class_name(submodule)
        for submodule in module.modules()
        if any(parameter.requires_grad for parameter in submodule.parameters(recurse=False))
    }
    return sorted(module_types)

def compatibility_result_path(project_root, requested_mode, target_epsilon, attempt_id):
    mode = parse_grad_sample_mode(requested_mode)
    attempt = parse_compatibility_attempt_id(attempt_id)
    epsilon_names = {0.5: "0.5", 2.0: "2", 8.0: "8"}
    if target_epsilon not in epsilon_names:
        raise ValueError("compatibility record epsilon must be 0.5, 2, or 8")
    return (
        Path(project_root).resolve()
        / "results/vaultgemma/compatibility"
        / f"{mode}-epsilon-{epsilon_names[target_epsilon]}"
        / f"{attempt}.json"
    )

def prepare_compatibility_parent(project_root, destination):
    root = Path(project_root).resolve()
    destination = Path(destination)
    if not destination.is_absolute():
        raise ValueError("compatibility destination must be absolute")
    try:
        relative_parent = destination.parent.relative_to(root)
    except ValueError as error:
        raise ValueError("compatibility destination escaped the project root") from error
    if not hasattr(os, "O_NOFOLLOW") or not hasattr(os, "O_DIRECTORY"):
        raise RuntimeError("safe compatibility directory walking is unavailable")
    flags = os.O_RDONLY | os.O_DIRECTORY | os.O_NOFOLLOW
    directory_descriptor = os.open(root, flags)
    try:
        for component in relative_parent.parts:
            if component in {"", ".", ".."}:
                raise ValueError("unsafe compatibility directory component")
            try:
                os.mkdir(component, mode=0o755, dir_fd=directory_descriptor)
            except FileExistsError:
                pass
            next_descriptor = os.open(component, flags, dir_fd=directory_descriptor)
            os.close(directory_descriptor)
            directory_descriptor = next_descriptor
        return directory_descriptor
    except BaseException:
        os.close(directory_descriptor)
        raise

def publish_compatibility_bytes_no_replace_at(parent_descriptor, destination_name, payload):
    if not isinstance(parent_descriptor, int) or parent_descriptor < 0:
        raise ValueError("compatibility parent descriptor must be open")
    if (
        not isinstance(destination_name, str)
        or destination_name in {"", ".", ".."}
        or Path(destination_name).name != destination_name
    ):
        raise ValueError("unsafe compatibility destination name")
    if not isinstance(payload, bytes):
        raise TypeError("compatibility payload must be bytes")
    temporary_descriptor = -1
    temporary_name = None
    try:
        temporary_flags = os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW
        for _ in range(128):
            candidate = f".{destination_name}.{os.urandom(16).hex()}.tmp"
            try:
                temporary_descriptor = os.open(
                    candidate, temporary_flags, 0o600, dir_fd=parent_descriptor
                )
            except FileExistsError:
                continue
            temporary_name = candidate
            break
        if temporary_descriptor < 0 or temporary_name is None:
            raise FileExistsError("could not allocate compatibility temporary file")
        remaining = memoryview(payload)
        while remaining:
            written = os.write(temporary_descriptor, remaining)
            if written <= 0:
                raise OSError("short compatibility record write")
            remaining = remaining[written:]
        os.fsync(temporary_descriptor)
        try:
            os.link(
                temporary_name, destination_name,
                src_dir_fd=parent_descriptor, dst_dir_fd=parent_descriptor,
                follow_symlinks=False,
            )
        except FileExistsError as error:
            existing_descriptor = -1
            try:
                existing_flags = os.O_RDONLY | os.O_NONBLOCK | os.O_NOFOLLOW
                existing_descriptor = os.open(
                    destination_name, existing_flags, dir_fd=parent_descriptor
                )
                if not stat.S_ISREG(os.fstat(existing_descriptor).st_mode):
                    raise FileExistsError("existing compatibility record is not regular")
                existing_chunks = []
                while True:
                    chunk = os.read(existing_descriptor, 1024 * 1024)
                    if not chunk:
                        break
                    existing_chunks.append(chunk)
            except FileExistsError:
                raise
            except OSError as inspection_error:
                raise FileExistsError(
                    "refusing to inspect or replace existing compatibility record"
                ) from inspection_error
            finally:
                if existing_descriptor >= 0:
                    os.close(existing_descriptor)
            if b"".join(existing_chunks) != payload:
                raise FileExistsError(
                    "refusing to overwrite mismatched compatibility record"
                ) from error
        else:
            os.fsync(parent_descriptor)
    finally:
        if temporary_descriptor >= 0:
            os.close(temporary_descriptor)
        if temporary_name is not None:
            try:
                os.unlink(temporary_name, dir_fd=parent_descriptor)
            except FileNotFoundError:
                pass

def is_lowercase_sha256(value):
    return (
        isinstance(value, str)
        and len(value) == 64
        and all(character in "0123456789abcdef" for character in value)
    )

def load_compatibility_provenance(project_root):
    root = Path(project_root).resolve()
    if not (root / "task_plan.md").is_file():
        raise RuntimeError("compatibility output requires the verified project root")
    workspace = root / "workspaces/vaultgemma_vectorized"
    upstream_path = workspace / "upstream_manifest.json"
    patch_path = workspace / "workspace_patch_manifest.json"
    if not upstream_path.is_file() or not patch_path.is_file():
        raise RuntimeError("compatibility output requires both provenance manifests")
    upstream_manifest = json.loads(upstream_path.read_text(encoding="utf-8"))
    patch_manifest = json.loads(patch_path.read_text(encoding="utf-8"))
    expected_upstream = {
        "repository_url": "https://github.com/google-gemini/gemma-cookbook.git",
        "commit": "3c2935f537ecb667ad8444490e13f1edcfef993c",
        "source_notebook": "Research/[VaultGemma]FineTuning_Inference_Huggingface.ipynb",
        "source_sha256": "46acb0beca164a9004de33b84961572eeca2fb099a675aa76225d34a1c33535d",
    }
    for field, expected_value in expected_upstream.items():
        if upstream_manifest.get(field) != expected_value:
            raise RuntimeError(f"official upstream provenance mismatch: {field}")
    if not is_lowercase_sha256(upstream_manifest["source_sha256"]):
        raise RuntimeError("upstream source hash is not lowercase SHA-256")
    current_sha256 = patch_manifest.get("current_notebook_sha256")
    if not is_lowercase_sha256(current_sha256):
        raise RuntimeError("current notebook hash is not lowercase SHA-256")
    current_notebook = workspace / "[VaultGemma]FineTuning_Vectorized.ipynb"
    if not current_notebook.is_file():
        raise RuntimeError("canonical workspace notebook is missing")
    actual_current_sha256 = hashlib.sha256(current_notebook.read_bytes()).hexdigest()
    if actual_current_sha256 != current_sha256:
        raise RuntimeError("current notebook bytes do not match provenance")
    upstream_source = root / "third_party/gemma-cookbook" / expected_upstream["source_notebook"]
    try:
        upstream_lstat = upstream_source.lstat()
    except FileNotFoundError as error:
        raise RuntimeError("official upstream source is missing") from error
    if stat.S_ISLNK(upstream_lstat.st_mode) or not stat.S_ISREG(upstream_lstat.st_mode):
        raise RuntimeError("official upstream source must be a regular non-symlink file")
    if not hasattr(os, "O_NOFOLLOW"):
        raise RuntimeError("safe upstream source inspection is unavailable")
    upstream_descriptor = -1
    try:
        upstream_descriptor = os.open(upstream_source, os.O_RDONLY | os.O_NOFOLLOW)
        upstream_fstat = os.fstat(upstream_descriptor)
        if (
            not stat.S_ISREG(upstream_fstat.st_mode)
            or upstream_fstat.st_dev != upstream_lstat.st_dev
            or upstream_fstat.st_ino != upstream_lstat.st_ino
        ):
            raise RuntimeError("official upstream source changed during inspection")
        upstream_digest = hashlib.sha256()
        while True:
            chunk = os.read(upstream_descriptor, 1024 * 1024)
            if not chunk:
                break
            upstream_digest.update(chunk)
        actual_upstream_sha256 = upstream_digest.hexdigest()
    except OSError as error:
        raise RuntimeError("official upstream source cannot be inspected safely") from error
    finally:
        if upstream_descriptor >= 0:
            os.close(upstream_descriptor)
    if actual_upstream_sha256 != expected_upstream["source_sha256"]:
        raise RuntimeError("official upstream source bytes do not match provenance")
    return {
        "upstream": dict(expected_upstream),
        "current_notebook_sha256": current_sha256,
    }

def safe_failure_message(error):
    del error
    return "compatibility gate failed; details redacted"

def validate_private_mode_result(
    requested_mode, private_objects, original_criterion, expected_batch_size,
    requested_module, base_optimizer, supplied_loader
):
    mode = parse_grad_sample_mode(requested_mode)
    expected_wrapper_types = {
        "hooks": GradSampleModule,
        "functorch": GradSampleModule,
        "ew": GradSampleModuleExpandedWeights,
        "ghost": GradSampleModuleFastGradientClipping,
    }
    expected_optimizer_types = {
        "hooks": DPOptimizer,
        "functorch": DPOptimizer,
        "ew": DPOptimizer,
        "ghost": DPOptimizerFastGradientClipping,
    }
    expected_arities = {"hooks": 3, "functorch": 3, "ew": 3, "ghost": 4}
    if not isinstance(private_objects, tuple) or len(private_objects) != expected_arities[mode]:
        raise TypeError(f"{mode} make_private returned the wrong tuple arity")
    private_module = private_objects[0]
    private_optimizer = private_objects[1]
    if type(private_module) is not expected_wrapper_types[mode]:
        raise TypeError(f"{mode} returned an unexpected private wrapper")
    if type(private_optimizer) is not expected_optimizer_types[mode]:
        raise TypeError(f"{mode} returned an unexpected private optimizer")
    private_loader = private_objects[3] if mode == "ghost" else private_objects[2]
    if private_module._module is not requested_module:
        raise ValueError(f"{mode} private wrapper does not contain the requested module")
    if private_optimizer.original_optimizer is not base_optimizer:
        raise ValueError(f"{mode} private optimizer does not contain the gate base optimizer")
    if private_loader is not supplied_loader:
        raise ValueError(f"{mode} returned loader is not the supplied loader")
    force_functorch = getattr(private_module, "force_functorch", None)
    expected_force_functorch = {
        "hooks": False, "functorch": True, "ew": None, "ghost": False
    }[mode]
    if force_functorch is not expected_force_functorch:
        raise ValueError(f"{mode} force_functorch flag mismatch")
    if private_optimizer.expected_batch_size != expected_batch_size:
        raise ValueError(f"{mode} expected batch size mismatch")
    if mode == "ghost":
        live_criterion = private_objects[2]
        if type(live_criterion) is not DPLossFastGradientClipping:
            raise TypeError("ghost returned an unexpected criterion wrapper")
        if live_criterion.module is not private_module:
            raise ValueError("ghost criterion is not bound to the live module")
        if live_criterion.optimizer is not private_optimizer:
            raise ValueError("ghost criterion is not bound to the live optimizer")
    else:
        live_criterion = original_criterion
    return {
        "actual_mode": mode,
        "wrapper_class": fully_qualified_class_name(private_module),
        "optimizer_class": fully_qualified_class_name(private_optimizer),
        "criterion_class": fully_qualified_class_name(live_criterion),
        "force_functorch": force_functorch,
        "private_return_arity": len(private_objects),
    }

def make_private_with_compatibility_gate(
    *, privacy_engine, module, optimizer_factory, criterion, data_loader,
    noise_multiplier, max_grad_norm, noise_generator, requested_mode,
    target_epsilon, target_delta, sample_rate, expected_batch_size, attempt_id, project_root
):
    mode = parse_grad_sample_mode(requested_mode)
    attempt = parse_compatibility_attempt_id(attempt_id)
    destination = compatibility_result_path(project_root, mode, target_epsilon, attempt)
    provenance = load_compatibility_provenance(project_root)
    trainable_module_types = collect_trainable_module_types(module)
    record = {
        "requested_mode": mode,
        "attempt_id": attempt,
        "poisson_sampling": False,
        "sample_rate": sample_rate,
        "noise_multiplier": noise_multiplier,
        "target_epsilon": target_epsilon,
        "target_delta": target_delta,
        "max_grad_norm": max_grad_norm,
        "expected_batch_size": expected_batch_size,
        "trainable_module_types": trainable_module_types,
        **provenance,
    }
    parent_descriptor = prepare_compatibility_parent(project_root, destination)
    failure_stage = "strict_module_validation"
    try:
        validation_errors = ModuleValidator.validate(module, strict=True)
        if validation_errors != []:
            raise RuntimeError(f"strict validation returned errors: {validation_errors!r}")
        failure_stage = "optimizer_creation"
        base_optimizer = optimizer_factory(module)
        failure_stage = "make_private"
        private_objects = privacy_engine.make_private(
            module=module,
            optimizer=base_optimizer,
            criterion=criterion,
            data_loader=data_loader,
            noise_multiplier=noise_multiplier,
            max_grad_norm=max_grad_norm,
            poisson_sampling=False,
            noise_generator=noise_generator,
            grad_sample_mode=mode,
        )
        failure_stage = "post_wrap_validation"
        expected_return_arity = 4 if mode == "ghost" else 3
        if not isinstance(private_objects, tuple) or len(private_objects) != expected_return_arity:
            raise TypeError(f"{mode} make_private returned the wrong tuple arity")
        private_objects[1].expected_batch_size = expected_batch_size
        private_objects[1].attach_step_hook(
            privacy_engine.accountant.get_optimizer_hook_fn(sample_rate=sample_rate)
        )
        actual = validate_private_mode_result(
            mode, private_objects, criterion, expected_batch_size,
            module, base_optimizer, data_loader
        )
        success_record = {**record, **actual, "status": "supported"}
        serialized = json.dumps(
            success_record, ensure_ascii=False, indent=2, sort_keys=True
        ) + "\n"
        failure_stage = "compatibility_recording"
        publish_compatibility_bytes_no_replace_at(
            parent_descriptor, destination.name, serialized.encode("utf-8")
        )
        return private_objects, success_record, destination
    except Exception as original_error:
        failure_record = {
            **record,
            "status": "failed",
            "actual_mode": None,
            "wrapper_class": None,
            "optimizer_class": None,
            "criterion_class": None,
            "force_functorch": None,
            "private_return_arity": None,
            "failure_stage": failure_stage,
            "exception": {
                "type": fully_qualified_class_name(original_error),
                "message": safe_failure_message(original_error),
            },
        }
        failure_serialized = json.dumps(
            failure_record, ensure_ascii=False, indent=2, sort_keys=True
        ) + "\n"
        try:
            publish_compatibility_bytes_no_replace_at(
                parent_descriptor, destination.name, failure_serialized.encode("utf-8")
            )
        except Exception as recording_error:
            combined = RuntimeError(
                f"compatibility stage {failure_stage} raised {type(original_error).__name__}; "
                f"recording also raised {type(recording_error).__name__}"
            )
            combined.original_error = original_error
            combined.recording_error = recording_error
            raise combined from recording_error
        raise original_error.with_traceback(original_error.__traceback__)
    finally:
        os.close(parent_descriptor)

TARGET_EPSILON_ENV = os.environ.get("VAULTGEMMA_TARGET_EPSILON")
TARGET_EPSILON = parse_target_epsilon(TARGET_EPSILON_ENV)
require_contract(Q == 128 / 7200, f"sampling-rate mismatch: {Q}")
require_contract(
    divmod(TRAIN_SIZE, LOGICAL_BATCH_SIZE)
    == (EXPECTED_FULL_LOGICAL_BATCHES_PER_EPOCH, EXPECTED_REMAINDER_RECORDS_PER_EPOCH),
    "logical-batch remainder contract mismatch",
)
require_contract(
    EXPECTED_LOGICAL_STEPS_PER_EPOCH * NUM_TRAIN_EPOCHS == TOTAL_OPTIMIZER_STEPS,
    "logical optimizer-step contract mismatch",
)
require_contract(
    EXPECTED_PHYSICAL_MICROBATCHES_PER_EPOCH * NUM_TRAIN_EPOCHS
    == EXPECTED_TOTAL_PHYSICAL_MICROBATCHES,
    "physical microbatch contract mismatch",
)

NOISE_MULTIPLIER = get_noise_multiplier(
    target_epsilon=TARGET_EPSILON,
    target_delta=TARGET_DELTA,
    sample_rate=Q,
    steps=TOTAL_OPTIMIZER_STEPS,
    accountant="prv",
)
require_contract(
    NOISE_MULTIPLIER == EXPECTED_NOISE_MULTIPLIERS[TARGET_EPSILON],
    f"Opacus PRV calibration mismatch: {NOISE_MULTIPLIER}",
)

# The compatibility gate validates strictly before its optimizer factory or wrapping.
peft_model.train()
DP_LOSS_CRITERION = torch.nn.CrossEntropyLoss(ignore_index=-100, reduction="mean")
privacy_engine = PrivacyEngine(accountant="prv")

def build_adamw_optimizer(private_model):
    return torch.optim.AdamW(private_model.parameters(), lr=learning_rate)

private_objects, COMPATIBILITY_RECORD, COMPATIBILITY_PATH = (
    make_private_with_compatibility_gate(
        privacy_engine=privacy_engine,
        module=peft_model,
        optimizer_factory=build_adamw_optimizer,
        criterion=DP_LOSS_CRITERION,
        data_loader=train_dataloader,
        noise_multiplier=NOISE_MULTIPLIER,
        max_grad_norm=MAX_GRAD_NORM,
        noise_generator=DP_NOISE_GENERATOR,
        requested_mode=GRAD_SAMPLE_MODE,
        target_epsilon=TARGET_EPSILON,
        target_delta=TARGET_DELTA,
        sample_rate=Q,
        expected_batch_size=LOGICAL_BATCH_SIZE,
        attempt_id=COMPATIBILITY_ATTEMPT_ID,
        project_root=project_root,
    )
)
if GRAD_SAMPLE_MODE == "ghost":
    require_contract(len(private_objects) == 4, "ghost make_private must return four objects")
    peft_model = private_objects[0]
    optimizer = private_objects[1]
    GHOST_LOSS_CRITERION = private_objects[2]
    train_dataloader = private_objects[3]
else:
    require_contract(len(private_objects) == 3, "non-ghost make_private must return three objects")
    peft_model = private_objects[0]
    optimizer = private_objects[1]
    train_dataloader = private_objects[2]
    GHOST_LOSS_CRITERION = None

# make_private derives 126 from 7200/57; enforce the fixed normalization denominator.
optimizer.expected_batch_size = LOGICAL_BATCH_SIZE
require_contract(
    optimizer.expected_batch_size == LOGICAL_BATCH_SIZE,
    "DP optimizer expected batch size mismatch",
)
# Learning rate scheduler advances only on completed logical optimizer steps.
num_training_steps = TOTAL_OPTIMIZER_STEPS
num_warmup_steps = NUM_WARMUP_STEPS
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=NUM_WARMUP_STEPS,
    num_training_steps=TOTAL_OPTIMIZER_STEPS,
)

print(
    f"Using explicit PRV sigma={NOISE_MULTIPLIER} for epsilon={TARGET_EPSILON}, "
    f"delta={TARGET_DELTA}, q={Q}, steps={TOTAL_OPTIMIZER_STEPS}."
)

## 8. Training Loop

The main training loop with the following features:
- **Gradient accumulation**: Accumulates gradients over 8 steps before updating weights
- **Automatic checkpointing**: Saves model when training loss drops below 0.06
- **Periodic validation**: Evaluates on validation set every 200 steps
- **Progress tracking**: Uses tqdm for visual progress bar

The loop will run for 2 epochs, logging metrics every 20 steps and evaluating every 200 steps.
Models are saved to the specified directory when performance thresholds are met.


In [ ]:
import resource
import time

def required_runner_value(name):
    value = os.environ.get(name)
    if not value:
        raise ValueError(f"{name} is required for runner execution")
    return value

def require_finite_metric(name, value, *, nonnegative):
    require_contract(
        isinstance(value, (int, float)) and not isinstance(value, bool),
        f"{name} must be numeric",
    )
    require_contract(math.isfinite(float(value)), f"{name} must be finite")
    if nonnegative:
        require_contract(value >= 0, f"{name} must be nonnegative")

def require_nonnegative_integer_metric(name, value):
    require_contract(
        isinstance(value, int) and not isinstance(value, bool) and value >= 0,
        f"{name} must be a nonnegative integer",
    )

def count_shifted_response_tokens_from_cpu(labels):
    require_contract(isinstance(labels, torch.Tensor), "labels must be a tensor")
    require_contract(labels.device.type == "cpu", "response tokens must be counted on CPU")
    require_contract(labels.ndim == 2 and labels.shape[1] >= 2, "labels must have [batch, sequence] shape")
    return int(labels[:, 1:].ne(-100).sum(dtype=torch.int64))

def summarize_benchmark_steps(records, *, warmup_steps, logical_batch_size):
    require_contract(isinstance(records, list), "benchmark records must be a list")
    require_contract(0 <= warmup_steps < len(records), "invalid benchmark warmup window")
    for expected_step, record in enumerate(records, start=1):
        require_contract(
            isinstance(record, dict) and set(record) == {
                "logical_step", "latency_seconds", "examples", "response_tokens"
            },
            "malformed logical-step record",
        )
        require_contract(record["logical_step"] == expected_step, "logical-step sequence mismatch")
        require_contract(
            isinstance(record["latency_seconds"], (int, float))
            and not isinstance(record["latency_seconds"], bool)
            and math.isfinite(float(record["latency_seconds"]))
            and record["latency_seconds"] > 0,
            "logical-step latency must be finite and positive",
        )
        require_contract(
            isinstance(record["examples"], int)
            and not isinstance(record["examples"], bool)
            and record["examples"] == logical_batch_size,
            "benchmark logical step must contain one full logical batch",
        )
        require_contract(
            isinstance(record["response_tokens"], int)
            and not isinstance(record["response_tokens"], bool)
            and record["response_tokens"] >= 0,
            "response-token count must be a nonnegative integer",
        )
    steady_records = records[warmup_steps:]
    latencies = [float(record["latency_seconds"]) for record in steady_records]
    response_tokens = [record["response_tokens"] for record in steady_records]
    steady_seconds = sum(latencies)
    steady_examples = sum(record["examples"] for record in steady_records)
    steady_response_tokens = sum(response_tokens)
    require_contract(steady_seconds > 0, "steady-state duration must be positive")
    return {
        "warmup_steps_excluded": warmup_steps,
        "steady_step_count": len(steady_records),
        "step_latency_seconds": latencies,
        "response_token_counts": response_tokens,
        "optimizer_step_seconds_p50": float(np.percentile(latencies, 50)),
        "optimizer_step_seconds_p95": float(np.percentile(latencies, 95)),
        "examples_per_second": steady_examples / steady_seconds,
        "response_tokens_per_second": steady_response_tokens / steady_seconds,
    }

BENCHMARK_LOGICAL_STEPS = 15
BENCHMARK_WARMUP_STEPS = 5
RUN_ID = parse_compatibility_attempt_id(required_runner_value("AI_SAFETY_RUN_ID"))
require_contract(RUN_ID == COMPATIBILITY_ATTEMPT_ID, "run and compatibility IDs differ")
RUN_KIND = required_runner_value("AI_SAFETY_RUN_KIND")
require_contract(RUN_KIND in {"full", "smoke", "benchmark"}, "invalid AI_SAFETY_RUN_KIND")
METRICS_PATH = Path(required_runner_value("AI_SAFETY_METRICS_FILE"))
ADAPTER_PATH = Path(required_runner_value("AI_SAFETY_ADAPTER_DIR"))
EXECUTED_INPUT_SHA256 = required_runner_value("AI_SAFETY_EXECUTED_INPUT_SHA256")
require_contract(
    re.fullmatch(r"[0-9a-f]{64}", EXECUTED_INPUT_SHA256) is not None,
    "executed input SHA-256 is invalid",
)
require_contract(METRICS_PATH.is_absolute(), "metrics path must be absolute")
require_contract(ADAPTER_PATH.is_absolute(), "adapter path must be absolute")
require_contract(
    METRICS_PATH == project_root / f"results/vaultgemma/runs/{RUN_ID}.json",
    "metrics path does not match the run ID",
)
max_updates_raw = required_runner_value("AI_SAFETY_MAX_UPDATES")
require_contract(
    re.fullmatch(r"0|[1-9][0-9]*", max_updates_raw) is not None,
    "AI_SAFETY_MAX_UPDATES must be canonical",
)
MAX_UPDATES = int(max_updates_raw)
if RUN_KIND == "full":
    require_contract(MAX_UPDATES == 0, "full runs require max_updates 0")
elif RUN_KIND == "benchmark":
    require_contract(MAX_UPDATES == BENCHMARK_LOGICAL_STEPS, "benchmark requires 15 updates")
    require_contract(TARGET_EPSILON == 2.0, "benchmark requires epsilon 2")
else:
    require_contract(0 < MAX_UPDATES <= TOTAL_OPTIMIZER_STEPS, "invalid smoke update limit")
RUN_TARGET_LOGICAL_STEPS = TOTAL_OPTIMIZER_STEPS if MAX_UPDATES == 0 else MAX_UPDATES
initialization_seconds = time.perf_counter() - NOTEBOOK_INITIALIZATION_STARTED
torch.cuda.reset_peak_memory_stats(device)
run_started = time.perf_counter()
train_started = time.perf_counter()

print("Starting training loop...")
require_contract(GRADIENT_ACCUMULATION_STEPS == 1, "gradient accumulation must remain 1")
progress_bar = tqdm(range(RUN_TARGET_LOGICAL_STEPS))
global_step = 0
scheduler_step_count = 0
physical_microbatch_count = 0
completed_epoch_count = 0
train_loss_record_sum = 0.0
train_record_count = 0
all_train_loss_record_sum = 0.0
all_train_record_count = 0
logical_step_started = None
logical_step_examples = 0
logical_step_response_tokens = 0
logical_step_records = []

for epoch in range(NUM_TRAIN_EPOCHS):
    peft_model.train()
    epoch_logical_step_start = global_step
    epoch_physical_step_start = physical_microbatch_count
    with BatchMemoryManager(
        data_loader=train_dataloader,
        max_physical_batch_size=MAX_PHYSICAL_BATCH_SIZE,
        optimizer=optimizer,
    ) as memory_safe_data_loader:
        for step, batch in enumerate(memory_safe_data_loader):
            physical_microbatch_count += 1
            cpu_batch_examples = int(batch["labels"].shape[0])
            cpu_response_tokens = count_shifted_response_tokens_from_cpu(batch["labels"])
            logical_step_examples += cpu_batch_examples
            logical_step_response_tokens += cpu_response_tokens
            if logical_step_started is None:
                torch.cuda.synchronize(device)
                logical_step_started = time.perf_counter()
            # Move one physical microbatch to the selected device.
            batch = {k: v.to(device) for k, v in batch.items()}

            # Physical-chunk means are accumulated with record counts.
            loss = forward_response_only_loss(
                peft_model, batch, optimizer, GRAD_SAMPLE_MODE, training=True,
                ghost_loss_criterion=GHOST_LOSS_CRITERION
            )
            train_loss_record_sum, train_record_count = accumulate_record_weighted_loss(
                train_loss_record_sum, train_record_count,
                loss.item(), batch["labels"].shape[0]
            )
            all_train_loss_record_sum, all_train_record_count = accumulate_record_weighted_loss(
                all_train_loss_record_sum, all_train_record_count,
                loss.item(), batch["labels"].shape[0]
            )
            loss.backward()

            # BMM signals skipped physical chunks to the DP optimizer.
            optimizer.step()
            logical_step_completed = optimizer_completed_logical_step(optimizer)
            optimizer.zero_grad()

            if logical_step_completed:
                torch.cuda.synchronize(device)
                logical_step_latency = time.perf_counter() - logical_step_started
                lr_scheduler.step()
                scheduler_step_count += 1
                global_step += 1
                logical_step_records.append({
                    "logical_step": global_step,
                    "latency_seconds": logical_step_latency,
                    "examples": logical_step_examples,
                    "response_tokens": logical_step_response_tokens,
                })
                logical_step_started = None
                logical_step_examples = 0
                logical_step_response_tokens = 0
                progress_bar.update(1)
                if RUN_KIND in {"smoke", "benchmark"} and global_step >= RUN_TARGET_LOGICAL_STEPS:
                    break

                # Logging, evaluation, and checkpointing use logical steps only.
                if global_step % logging_steps == 0:
                    avg_train_loss = finalize_record_weighted_loss(
                        train_loss_record_sum, train_record_count
                    )
                    log_message = f"Step {global_step}: Train Loss = {avg_train_loss:.4f}"

                    if global_step % eval_steps == 0:
                        peft_model.eval()
                        eval_loss_record_sum = 0.0
                        eval_record_count = 0
                        with torch.no_grad():
                            for eval_batch in eval_dataloader:
                                eval_batch = {k: v.to(device) for k, v in eval_batch.items()}
                                eval_loss = forward_response_only_loss(
                                    peft_model, eval_batch, optimizer, GRAD_SAMPLE_MODE, training=False,
                                    ghost_loss_criterion=GHOST_LOSS_CRITERION
                                )
                                eval_loss_record_sum, eval_record_count = accumulate_record_weighted_loss(
                                    eval_loss_record_sum, eval_record_count,
                                    eval_loss.item(), eval_batch["labels"].shape[0]
                                )
                        avg_eval_loss = finalize_record_weighted_loss(
                            eval_loss_record_sum, eval_record_count
                        )
                        log_message += f" | Validation Loss = {avg_eval_loss:.4f}"
                        peft_model.train()

                    print(log_message)
                    train_loss_record_sum = 0.0
                    train_record_count = 0

    if RUN_KIND in {"smoke", "benchmark"} and global_step >= RUN_TARGET_LOGICAL_STEPS:
        break

    require_contract(
        global_step - epoch_logical_step_start == EXPECTED_LOGICAL_STEPS_PER_EPOCH,
        f"epoch {epoch} logical-step mismatch",
    )
    require_contract(
        physical_microbatch_count - epoch_physical_step_start
        == EXPECTED_PHYSICAL_MICROBATCHES_PER_EPOCH,
        f"epoch {epoch} physical-microbatch mismatch",
    )
    completed_epoch_count += 1

progress_bar.close()
train_seconds = time.perf_counter() - train_started
train_only_seconds = sum(record["latency_seconds"] for record in logical_step_records)
require_contract(global_step == RUN_TARGET_LOGICAL_STEPS, "final logical-step count mismatch")
require_contract(len(logical_step_records) == global_step, "logical-step record count mismatch")
require_contract(logical_step_started is None, "logical-step timer remained active")
require_contract(scheduler_step_count == RUN_TARGET_LOGICAL_STEPS, "scheduler-step count mismatch")
require_contract(lr_scheduler.last_epoch == RUN_TARGET_LOGICAL_STEPS, "scheduler last_epoch mismatch")
require_contract(
    accountant_logical_steps(privacy_engine.accountant) == RUN_TARGET_LOGICAL_STEPS,
    "PRV accountant logical-step count mismatch",
)
if RUN_KIND == "full":
    require_contract(completed_epoch_count == NUM_TRAIN_EPOCHS, "completed-epoch count mismatch")
    require_contract(
        physical_microbatch_count == EXPECTED_TOTAL_PHYSICAL_MICROBATCHES,
        "total physical-microbatch count mismatch",
    )
else:
    require_contract(physical_microbatch_count > 0, "bounded run processed no physical microbatches")
steady_state_metrics = None
if RUN_KIND == "benchmark":
    steady_state_metrics = summarize_benchmark_steps(
        logical_step_records,
        warmup_steps=BENCHMARK_WARMUP_STEPS,
        logical_batch_size=LOGICAL_BATCH_SIZE,
    )
    require_contract(
        steady_state_metrics["steady_step_count"] == 10,
        "benchmark must retain exactly 10 steady-state steps",
    )
require_contract(optimizer.expected_batch_size == LOGICAL_BATCH_SIZE, "final expected batch mismatch")
require_contract(optimizer.noise_multiplier == NOISE_MULTIPLIER, "final noise multiplier mismatch")
require_contract(TARGET_EPSILON in ALLOWED_TARGET_EPSILONS, "final epsilon selection mismatch")
require_contract(Q == 128 / 7200, "final accountant sampling rate mismatch")
validate_accountant_history(
    privacy_engine.accountant, NOISE_MULTIPLIER, Q, RUN_TARGET_LOGICAL_STEPS
)

# Final requested Opacus PRV approximation for fixed-size non-Poisson sampling.
epsilon = privacy_engine.get_epsilon(delta=TARGET_DELTA)
require_contract(epsilon <= TARGET_EPSILON, "calibrated epsilon exceeds its target")
if RUN_KIND == "full":
    require_contract(
        TARGET_EPSILON - epsilon <= EPSILON_CALIBRATION_TOLERANCE,
        "calibrated epsilon is outside the Opacus tolerance",
    )
print(
    f"Opacus PRV approximation for fixed-size non-Poisson sampling: "
    f"ε = {epsilon:.2f} for δ = {TARGET_DELTA}"
)

eval_started = time.perf_counter()
eval_accountant_steps_before = accountant_logical_steps(privacy_engine.accountant)
eval_scheduler_steps_before = scheduler_step_count
optimizer.zero_grad(set_to_none=True)
peft_model.eval()
eval_loss_record_sum = 0.0
eval_record_count = 0
with torch.no_grad():
    for eval_batch in eval_dataloader:
        eval_batch = {key: value.to(device) for key, value in eval_batch.items()}
        eval_loss = forward_response_only_loss(
            peft_model, eval_batch, optimizer, GRAD_SAMPLE_MODE, training=False,
            ghost_loss_criterion=GHOST_LOSS_CRITERION,
        )
        eval_loss_record_sum, eval_record_count = accumulate_record_weighted_loss(
            eval_loss_record_sum, eval_record_count,
            eval_loss.item(), eval_batch["labels"].shape[0],
        )
require_contract(eval_record_count == EVAL_SIZE, "final eval record count mismatch")
require_contract(
    accountant_logical_steps(privacy_engine.accountant) == eval_accountant_steps_before,
    "final eval changed accountant steps",
)
require_contract(
    scheduler_step_count == eval_scheduler_steps_before,
    "final eval changed scheduler steps",
)
final_eval_loss = finalize_record_weighted_loss(eval_loss_record_sum, eval_record_count)
eval_seconds = time.perf_counter() - eval_started
final_train_loss = finalize_record_weighted_loss(
    all_train_loss_record_sum, all_train_record_count
)
final_perplexity = math.exp(final_eval_loss) if final_eval_loss < 700 else math.inf

require_contract(ADAPTER_PATH.is_dir(), "runner-owned adapter directory is missing")
require_contract(not any(ADAPTER_PATH.iterdir()), "adapter directory is not empty")
checkpoint_started = time.perf_counter()
save_private_adapter_checkpoint(peft_model, tokenizer, ADAPTER_PATH)
checkpoint_seconds = time.perf_counter() - checkpoint_started

run_seconds = time.perf_counter() - run_started
peak_allocated_bytes = torch.cuda.max_memory_allocated(device)
peak_reserved_bytes = torch.cuda.max_memory_reserved(device)
notebook_process_peak_rss_bytes = int(
    resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024
)
require_finite_metric("final_train_loss", final_train_loss, nonnegative=True)
require_finite_metric("final_eval_loss", final_eval_loss, nonnegative=True)
require_finite_metric("perplexity", final_perplexity, nonnegative=True)
require_finite_metric("final_epsilon", epsilon, nonnegative=True)
require_finite_metric("noise_multiplier", NOISE_MULTIPLIER, nonnegative=True)
require_finite_metric("notebook_initialization_seconds_from_import_cell", initialization_seconds, nonnegative=True)
require_finite_metric("train_seconds", train_seconds, nonnegative=True)
require_finite_metric("train_only_seconds", train_only_seconds, nonnegative=True)
require_finite_metric("eval_seconds", eval_seconds, nonnegative=True)
require_finite_metric("checkpoint_seconds", checkpoint_seconds, nonnegative=True)
require_finite_metric("run_seconds", run_seconds, nonnegative=True)
require_nonnegative_integer_metric("allocated_bytes", peak_allocated_bytes)
require_nonnegative_integer_metric("reserved_bytes", peak_reserved_bytes)
require_nonnegative_integer_metric("notebook_process_peak_rss_bytes", notebook_process_peak_rss_bytes)
require_nonnegative_integer_metric("logical_total_parameters", total_parameters)
require_nonnegative_integer_metric("logical_trainable_parameters", trainable_parameters)
require_finite_metric("logical_trainable_percent", trainable_percent, nonnegative=True)
require_nonnegative_integer_metric("packed_parameter_elements", packed_parameter_elements)
require_nonnegative_integer_metric("packed_storage_bytes", packed_storage_bytes)
if steady_state_metrics is not None:
    require_finite_metric("optimizer_step_seconds_p50", steady_state_metrics["optimizer_step_seconds_p50"], nonnegative=True)
    require_finite_metric("optimizer_step_seconds_p95", steady_state_metrics["optimizer_step_seconds_p95"], nonnegative=True)
    require_finite_metric("examples_per_second", steady_state_metrics["examples_per_second"], nonnegative=True)
    require_finite_metric("response_tokens_per_second", steady_state_metrics["response_tokens_per_second"], nonnegative=True)

provenance = dict(load_compatibility_provenance(project_root))
provenance["executed_input_sha256"] = EXECUTED_INPUT_SHA256
actual_mode = COMPATIBILITY_RECORD["actual_mode"]
require_contract(actual_mode == GRAD_SAMPLE_MODE, "actual mode differs from requested mode")
metrics = {
    "schema_version": 1,
    "run_id": RUN_ID,
    "status": "success",
    "configuration": {
        "run_kind": RUN_KIND,
        "model_id": MODEL_ID,
        "dataset_id": DATASET_ID,
        "seed": SEED,
        "target_epsilon": TARGET_EPSILON,
        "target_delta": TARGET_DELTA,
        "sample_rate": Q,
        "logical_batch_size": LOGICAL_BATCH_SIZE,
        "max_physical_batch_size": MAX_PHYSICAL_BATCH_SIZE,
        "max_updates": MAX_UPDATES,
    },
    "model_parameters": {
        "logical_total": total_parameters,
        "logical_trainable": trainable_parameters,
        "logical_trainable_percent": trainable_percent,
        "packed_physical_numel": packed_parameter_elements,
        "packed_storage_bytes": packed_storage_bytes,
    },
    "upstream_hashes": provenance,
    "executed_input_sha256": EXECUTED_INPUT_SHA256,
    "dataset_hash": selected_content_sha256,
    "requested_mode": GRAD_SAMPLE_MODE,
    "actual_mode": actual_mode,
    "noise_multiplier": NOISE_MULTIPLIER,
    "final_epsilon": epsilon,
    "logical_steps": global_step,
    "physical_steps": physical_microbatch_count,
    "train_loss": final_train_loss,
    "eval_loss": final_eval_loss,
    "perplexity": final_perplexity,
    "logical_optimizer_steps": logical_step_records,
    "steady_state": steady_state_metrics,
    "timings": {
        "notebook_initialization_seconds_from_import_cell": initialization_seconds,
        "train_seconds": train_seconds,
        "train_only_seconds": train_only_seconds,
        "eval_seconds": eval_seconds,
        "checkpoint_seconds": checkpoint_seconds,
        "run_seconds": run_seconds,
    },
    "pytorch_peak_memory": {
        "scope": "training_eval_checkpoint_after_reset",
        "allocated_bytes": peak_allocated_bytes,
        "reserved_bytes": peak_reserved_bytes,
    },
    "cpu_memory": {
        "notebook_process_peak_rss_bytes": notebook_process_peak_rss_bytes,
    },
    "checkpoint_path": str(ADAPTER_PATH),
}
serialized_metrics = json.dumps(
    metrics, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False
) + "\n"
publish_bytes_no_replace(METRICS_PATH, serialized_metrics.encode("utf-8"))
print(
    f"Completed {RUN_KIND} run {RUN_ID}: mode={actual_mode}, "
    f"logical_steps={global_step}, epsilon={epsilon:.6f}"
)

In [ ]:
model_path = kagglehub.model_download("google/vaultgemma/transformers/1b")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map="auto",
)

adapter_path = "./final_model"

tokenizer = GemmaTokenizer.from_pretrained(adapter_path)
tokenizer.pad_token = tokenizer.eos_token

peft_model = PeftModel.from_pretrained(base_model, adapter_path, is_trainable=False)

peft_model.eval()


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): VaultGemmaForCausalLM(
      (model): VaultGemmaModel(
        (embed_tokens): Embedding(256000, 1152, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x VaultGemmaDecoderLayer(
            (self_attn): VaultGemmaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1152, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1152, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1024, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
           

In [ ]:
def generate_response(question, max_new_tokens=128, temperature=0.9, top_p=0.9):
    prompt = f"Instruction:\nAnswer this question truthfully.\n\nQuestion:\n{question}\n\nResponse:\n"
    
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    inputs = {k: v.to(peft_model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.4 
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "Response:" in full_response:
        response = full_response.split("Response:")[-1].strip()
    else:
        response = full_response
    
    return response


question = "What is the role of insulin in the human body?"

response = generate_response(question)
print(f"\nQuestion: {question}")
print(f"Answer: {response}")



Question: What is the role of insulin in the human body?
Answer: The hormone affects several important functions including its actions on blood glucose levels which control hormones to lower or maintain it so that we can feel more adequate and are able respond better
